In [1]:
import pandas as pd

df = pd.read_excel("../raw/Superstore.xls", sheet_name="Orders")

base = {
    "rows": len(df),
    "sales": round(df["Sales"].sum(), 2),
    "profit": round(df["Profit"].sum(), 2),
}
base

FileNotFoundError: [Errno 2] No such file or directory: '../raw/Superstore.xls'

In [ ]:
df.columns = (df.columns.str.strip().str.lower()
              .str.replace("/", "_", regex=False)
              .str.replace("-", "_", regex=False)
              .str.replace(" ", "_", regex=False))
print(df.columns.tolist())

['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode', 'customer_id', 'customer_name', 'segment', 'country_region', 'city', 'state_province', 'postal_code', 'region', 'product_id', 'category', 'sub_category', 'product_name', 'sales', 'quantity', 'discount', 'profit']


In [ ]:
df["postal_code"] = df["postal_code"].astype(str).str.strip()
us = df["country_region"] == "United States"
df.loc[us, "postal_code"] = df.loc[us, "postal_code"].str.zfill(5)

print(df.loc[us, "postal_code"].str.len().value_counts())
print(df.loc[~us, "postal_code"].str.len().value_counts())

postal_code
5    9994
Name: count, dtype: int64
postal_code
3    200
Name: count, dtype: int64


In [ ]:
df["order_year"] = df["order_date"].dt.year
df["order_month"] = df["order_date"].dt.month
df["year_month"] = df["order_date"].dt.strftime("%Y-%m")
df["is_loss"] = (df["profit"] < 0).astype(int)
df["discount_band"] = pd.cut(
    df["discount"],
    bins=[-0.01, 0, 0.10, 0.20, 0.30, 0.40, 1.00],
    labels=["0%", "1-10%", "11-20%", "21-30%", "31-40%", ">40%"]
).astype(str)

In [ ]:
assert len(df) == base["rows"]
assert round(df["sales"].sum(), 2) == base["sales"]
assert round(df["profit"].sum(), 2) == base["profit"]
print("Loss rows:", df["is_loss"].sum())
print(df["discount_band"].value_counts())
print(df.isnull().sum().sum(), "missing values")

Loss rows: 1901
discount_band
0%        4925
11-20%    3758
>40%       951
31-40%     234
21-30%     230
1-10%       96
Name: count, dtype: int64
0 missing values


In [ ]:
import os
os.makedirs("../clean", exist_ok=True)

df.to_csv("../clean/superstore_clean.csv", index=False)
df.to_excel("../clean/superstore_clean.xlsx", index=False)

check = pd.read_excel("../clean/superstore_clean.xlsx")
print("Saved. Shape:", check.shape)

Saved. Shape: (10194, 26)


In [ ]:
import pandas as pd

df = pd.read_csv("../data/clean/superstore_clean.csv", dtype={"postal_code": str})
print(df.shape)   # must show (10194, 26)

df.to_excel("../excel/profitability_analysis.xlsx", sheet_name="Data", index=False)
print("Saved")

(10194, 26)
Saved


In [ ]:
import pandas as pd

c = pd.read_csv("../data/clean/superstore_clean.csv", dtype={"postal_code": str})
x = pd.read_excel("../excel/profitability_analysis.xlsx", sheet_name="Data")

print("CSV:", c.shape, "| Excel:", x.shape)

missing = c[~c["row_id"].isin(x["row_id"])]
display(missing[["row_id", "order_id", "order_date", "product_name", "sales", "profit"]])

CSV: (10194, 26) | Excel: (10194, 26)


,row_id,order_id,order_date,product_name,sales,profit


In [ ]:
print("CSV   sales:", round(c["sales"].sum(), 2), "| profit:", round(c["profit"].sum(), 2))
print("Excel sales:", round(x["sales"].sum(), 2), "| profit:", round(x["profit"].sum(), 2))

CSV   sales: 2326534.35 | profit: 292296.81
Excel sales: 2326534.35 | profit: 292296.81


In [1]:
import pandas as pd
df = pd.read_excel("../data/raw/Superstore.xls", sheet_name="Orders")
print(df.shape)

(10194, 21)
